# Baseline Models

In [1]:
import sys
import numpy as np
import pandas as pd

sys.path.append(r"C:\Users\arbaz2\Desktop\Quant Finance\Volatility Forecasting\src")
from data_pipeline import make_dataset

CRISIS_WINDOWS = {
    "GFC_2007_2009": ("2007-07-01", "2009-06-30"),
    "COVID_2020": ("2020-02-15", "2020-05-31"),
}

df = make_dataset(["SPY", "JPM"], start="2000-01-01", horizons=(1,5), crisis_windows=CRISIS_WINDOWS)
df["date"] = pd.to_datetime(df["date"])
df.head()

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,rv1_var,rv5_var,regime
0,2000-01-10,JPM,48.500000,48.916668,47.666668,47.666668,22.389442,4723500,-1.733130,3.003741,5.704533,14.393708,calm
1,2000-01-10,SPY,146.250000,146.906250,145.031250,146.250000,91.641869,5741700,0.342453,0.117274,1.449179,36.091892,calm
2,2000-01-11,JPM,46.666668,46.958332,45.500000,46.541668,21.861025,8405550,-2.388416,5.704533,0.390240,14.400442,calm
3,2000-01-11,SPY,145.812500,146.093750,143.500000,144.500000,90.545280,7503700,-1.203819,1.449179,0.999437,37.059380,calm
4,2000-01-12,JPM,46.458332,47.250000,46.333332,46.833332,21.998016,7271850,0.624692,0.390240,2.253423,14.666425,calm


## 1. Metrics (Variance Forecasting)

### QLIKE (Quasi-likelihood) is the standard loss for variance/volatility forecasts that behaves like a Gaussian log-likelihood and stays well-behaved even when your "true" variance is noisy:

$$ QLIKE( y , \hat y) = \log( \hat y) + \frac{y}{\hat y} $$

##### (up to an additive constant; lower is better)

* It penalizes you for predicting variance that's too small (because $ \frac{y}{\hat y} $ blows up)
* It also penalizes you for predicting variance that's too large (because $ \log ( \hat y)$ increases)

When realized variance is a noisy proxy for the true variance, QLIKE is relatively robust: models ranked best by QLIKE tend to stay best even if the proxy is imperfect.
It's also scale aware: it cares about proportional errors in variance, not just absolute differences.
Lower average QLIKE over out-of-sample period means a better variance forecast

In [2]:
def qlike(y_true, y_pred, eps=1e-12):
    y_pred = np.maximum(y_pred, eps)
    y_true = np.maximum(y_true, eps)
    return np.log(y_pred) + (y_true / y_pred)


# Function to calculate Mean Square Error (MSE)
def mse(y_true, y_pred):
    return (y_true - y_pred) ** 2

## Baseline Models

### Historical Variance (Rolling window)
#### Forecast nex-day variance as rolling variance of returns

#### For 5-day variance a simple baseline is:
$$ \hat {RV^{(5)}_t} \approx 5. \hat {\sigma^2_t}

In [3]:
def hv_forecast_var(d, window=20):
    """
    d: DataFrame for a single ticker, sorted by date, must include 'ret'
    Returns a Series of 1-day-ahead variance forecasts aligned at time t:
      hv1_var(t) = Var(ret_{t-window+1}...ret_t)
    """
    return d["ret"].rolling(window).var()

def hv_forecast_var_h(d, window=20, horizon=5):
    return horizon * hv_forecast_var(d, window=window)

### EWMA variance (RiskMetrics)

$$ \sigma^2_t = \lambda \sigma^2_{t-1} + ( 1 - \lambda) r^2_{t-1} $$

It produces a time series of variance forecasts update recursively

In [4]:
def ewma_var_series(ret, lam=0.94): # Taking lambda = 0.94 here
    """
    ret: pd.Series of returns (percent), indexed by date
    Returns sigma2_t aligned at time t (uses info up to t-1 for update).
    """
    r2 = ret**2
    sigma2 = np.empty(len(ret), dtype= float)
    sigma2[:] = np.nan
    
    # initialize with sample variance (can also use r2.mean())
    sigma2[0] = np.nanvar(ret.values, ddof=1)

    for t in range(1, len(ret)):
        sigma2[t] = lam * sigma2[t-1] + (1 - lam) * r2.iloc[t-1]

    return pd.Series(sigma2, index=ret.index)

def ewma_forecast_var_h(d, lam=0.94, horizon=1):
    """
    d: single ticker df sorted by date
    Returns horizon-ahead variance forecast aligned at time t.
    For horizon=5, a simple approximation is 5 * sigma2_t.
    """
    sigma2 = ewma_var_series(d["ret"], lam=lam)
    return horizon * sigma2

### Walk-forward evaluation.
We'll do
* Expanding training window (implicitly, since baselines don't "train")
* out-of-sample evaluation after a start date

In [5]:
EVAL_START = "2005-01-01"

#### Build forecasts for each ticker

In [6]:
def ewma_var_from_ret(ret: pd.Series, lam=0.94) -> pd.Series:
    """
    EWMA variance series sigma2_t aligned at time t (uses ret_{t-1}^2 in update).
    ret is a Series for ONE ticker.
    """
    r2 = ret**2
    sigma2 = np.empty(len(ret), dtype=float)
    sigma2[:] = np.nan

    # Initialize with sample variance of the series
    sigma2[0] = np.nanvar(ret.values, ddof=1)

    for t in range(1, len(ret)):
        sigma2[t] = lam * sigma2[t-1] + (1 - lam) * r2.iloc[t-1]

    return pd.Series(sigma2, index=ret.index)

def make_baseline_forecasts(df: pd.DataFrame, hv_window=20, ewma_lam=0.94) -> pd.DataFrame:
    """
    Add baseline variance forecasts to the panel dataframe (no pandas apply warnings):
      - hv1_var, hv5_var
      - ewma1_var, ewma5_var

    Notes:
    - HV uses rolling variance of returns.
    - EWMA uses RiskMetrics recursion.
    - 5-day forecasts are approximated as 5 * 1-day conditional variance.
    """
    out = df.copy().sort_values(["ticker", "date"]).reset_index(drop=True)

    # Rolling Historical Variance (1-day variance forecast)
    out["hv1_var"] = out.groupby("ticker")["ret"].transform(
        lambda s: s.rolling(hv_window).var()
    )
    out["hv5_var"] = 5.0 * out["hv1_var"]

    # EWMA conditional variance series (1-day variance forecast)
    out["ewma1_var"] = out.groupby("ticker")["ret"].transform(
        lambda s: ewma_var_from_ret(s, lam=ewma_lam)
    )
    out["ewma5_var"] = 5.0 * out["ewma1_var"]

    # Return in panel-friendly order
    out = out.sort_values(["date", "ticker"]).reset_index(drop=True)
    return out

In [7]:
df_f = make_baseline_forecasts(df, hv_window=20, ewma_lam=0.94)
df_f.head(40)

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,rv1_var,rv5_var,regime,hv1_var,hv5_var,ewma1_var,ewma5_var
0,2000-01-10,JPM,48.500000,48.916668,47.666668,47.666668,22.389442,4723500,-1.733130,3.003741,5.704533,14.393708,calm,NaN,NaN,5.291809,26.459044
1,2000-01-10,SPY,146.250000,146.906250,145.031250,146.250000,91.641869,5741700,0.342453,0.117274,1.449179,36.091892,calm,NaN,NaN,1.468518,7.342591
2,2000-01-11,JPM,46.666668,46.958332,45.500000,46.541668,21.861025,8405550,-2.388416,5.704533,0.390240,14.400442,calm,NaN,NaN,5.154525,25.772624
3,2000-01-11,SPY,145.812500,146.093750,143.500000,144.500000,90.545280,7503700,-1.203819,1.449179,0.999437,37.059380,calm,NaN,NaN,1.387443,6.937217
4,2000-01-12,JPM,46.458332,47.250000,46.333332,46.833332,21.998016,7271850,0.624692,0.390240,2.253423,14.666425,calm,NaN,NaN,5.187525,25.937626
5,2000-01-12,SPY,144.593750,144.593750,142.875000,143.062500,89.644592,6907700,-0.999718,0.999437,1.809371,36.243829,calm,NaN,NaN,1.391148,6.955738
6,2000-01-13,JPM,47.416668,48.333332,47.041668,47.541668,22.330729,6918900,1.501141,2.253423,12.463111,23.815049,calm,NaN,NaN,4.899688,24.498441
7,2000-01-13,SPY,144.468750,145.750000,143.281250,145.000000,90.858574,5158300,1.345129,1.809371,1.818882,6.194144,calm,NaN,NaN,1.367645,6.838225
8,2000-01-14,JPM,49.291668,50.500000,48.541668,49.250000,23.133154,9731850,3.530313,12.463111,15.756857,36.568165,calm,NaN,NaN,4.740912,23.704561
9,2000-01-14,SPY,146.531250,147.468750,145.968750,146.968750,92.092247,7437300,1.348659,1.818882,0.623929,6.700799,calm,NaN,NaN,1.394149,6.970743


hv1_var and hv5_var are NaN early on — that’s expected because rolling variance needs a full window (e.g., 20 days)

#### Score models overall and by regime

In [8]:
def score(df_f, model_cols, target_col, eval_start=EVAL_START):
    d = df_f[df_f["date"] >= pd.to_datetime(eval_start)].copy()

    rows = []
    for m in model_cols:
        valid = d.dropna(subset=[m, target_col])
        rows.append({
            "model": m,
            "target": target_col,
            "n": len(valid),
            "QLIKE": qlike(valid[target_col].values, valid[m].values).mean(),
            "MSE": mse(valid[target_col].values, valid[m].values).mean(),
        })
    return pd.DataFrame(rows).sort_values("QLIKE")

def score_by_regime(df_f, model_cols, target_col, eval_start=EVAL_START):
    d = df_f[df_f["date"] >= pd.to_datetime(eval_start)].copy()
    out = []
    for regime, g in d.groupby("regime"):
        for m in model_cols:
            valid = g.dropna(subset=[m, target_col])
            if len(valid) == 0:
                continue
            out.append({
                "regime": regime,
                "model": m,
                "target": target_col,
                "n": len(valid),
                "QLIKE": qlike(valid[target_col].values, valid[m].values).mean(),
                "MSE": mse(valid[target_col].values, valid[m].values).mean(),
            })
    return pd.DataFrame(out).sort_values(["target","regime","QLIKE"])

In [9]:
models_1d = ["hv1_var", "ewma1_var"]
models_5d = ["hv5_var", "ewma5_var"]

print(score(df_f, models_1d, "rv1_var"))
print(score(df_f, models_5d, "rv5_var"))

score_by_regime(df_f, models_1d, "rv1_var").head(20)

       model   target      n     QLIKE         MSE
1  ewma1_var  rv1_var  10888  1.445828  221.382484
0    hv1_var  rv1_var  10888  1.486513  230.501102
       model   target      n     QLIKE          MSE
0    hv5_var  rv5_var  10888  2.888738  1424.956574
1  ewma5_var  rv5_var  10888  2.909015  1302.073335


,regime,model,target,n,QLIKE,MSE
0,COVID_2020,hv1_var,rv1_var,144,4.259608,1823.978756
1,COVID_2020,ewma1_var,rv1_var,144,4.591994,1750.482479
3,GFC_2007_2009,ewma1_var,rv1_var,1008,3.034459,1938.600566
2,GFC_2007_2009,hv1_var,rv1_var,1008,3.092548,2022.667393
5,calm,ewma1_var,rv1_var,9736,1.234819,20.977161
4,calm,hv1_var,rv1_var,9736,1.279220,21.383970


* For 1-day result: ewma1_var QLIKE = 1.450 and hv1_var QLIKE = 1.490
  EWMA slighlty beats rolling HV which is consistent with theory as EWMA reacts faster to volalility clustering while HV is more sluggish.

* For 5-day result: ewma5_var QLIKE = 2.911 and hv5_var QLIKE = 2.890
  Now, HV slightly beats EWMA which makes sense as for longer horizons simple scaling (5* \sigma^2) is a rough approximation. Rolling window may stabilize better for multi-day horizon and EWMA may overreact to recent shocks, hurting longer-horizon aggregation.

  Different models behave differently across horizons.
  

* CALM Regime: ewma1_var QLIKE = 1.234 and hv1_var QLIKE = 1.278
  EWMA is better in calm. This is expected as volatility clustering persists in calm periods and EWMA adapts smoothly.

* Great Financial Crisis (GFC): ewma1_var QLIKE = 3.034 and hv1_var QLIKE = 3.092
  EWMA is slightly better. Interpretation: During prolonged crisis (GFC lasted long) both models struggle and EWMA adapts slighly faster.

* COVID: ewma1_var QLIKE = 4.591 and hv1_var QLIKE = 4.259
  Here HV wins. This is intersting and realistic as Covid was a sudden shock and EWMA heavily weights recent extreme returns and it may overshoot and then overestimate variance when shock subsides. Rolling window smooths slighly differently.
  This is a good "failure mode" example.


#### Score separately by ticker

In [10]:
def score_by_ticker(df_f, model_cols, target_col, eval_start=EVAL_START):
    d = df_f[df_f["date"] >= pd.to_datetime(eval_start)].copy()
    out = []
    for ticker, g in d.groupby("ticker"):
        for m in model_cols:
            valid = g.dropna(subset=[m, target_col])
            out.append({
                "ticker": ticker,
                "model": m,
                "target": target_col,
                "n": len(valid),
                "QLIKE": qlike(valid[target_col].values, valid[m].values).mean(),
                "MSE": mse(valid[target_col].values, valid[m].values).mean(),
            })
    return pd.DataFrame(out).sort_values(["ticker","target","QLIKE"])

score_by_ticker(df_f, models_1d, "rv1_var").head(10)

,ticker,model,target,n,QLIKE,MSE
1,JPM,ewma1_var,rv1_var,5444,1.992659,415.037595
0,JPM,hv1_var,rv1_var,5444,2.033360,432.115862
3,SPY,ewma1_var,rv1_var,5444,0.898997,27.727374
2,SPY,hv1_var,rv1_var,5444,0.939666,28.886343


For JPM QLIKE ~ 2.0 while for SPY QLIKE ~ 0.9 which makes sense as:
* For JPM we have higher idiosyncratic risk, larger jumps and is harder to forecast
* For SPY we have diversified index, smoother volatility dynamics and is easier to model.

#### These baseline models are univariate and are fitted independently for different tickers because of which adding more tickers does not improve the forecast.

### Visualization

In [11]:
''' Varinace Comparison'''

import plotly.graph_objects as go

def plot_forecast_vs_realized(df_f, ticker, model_col, target_col):
    d = df_f[df_f["ticker"] == ticker].sort_values("date").copy()

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=d["date"], y=d[target_col],
        mode="lines", name="Realized Variance"
    ))

    fig.add_trace(go.Scatter(
        x=d["date"], y=d[model_col],
        mode="lines", name=f"Forecast ({model_col})"
    ))

    fig.update_layout(
        title=f"{ticker}: {model_col} vs {target_col}",
        xaxis_title="Date",
        yaxis_title="Variance"
    )

    fig.show()

In [12]:
plot_forecast_vs_realized(df_f, "SPY", "ewma1_var", "rv1_var")

In [13]:
plot_forecast_vs_realized(df_f, "SPY", "hv1_var", "rv1_var")

In [14]:
''' Compare in Volatility space. This makes crisis spikes easier to interpret'''

def plot_vol_forecast(df_f, ticker, model_col, target_col):
    d = df_f[df_f["ticker"] == ticker].sort_values("date").copy()

    d["realized_vol"] = np.sqrt(d[target_col])
    d["forecast_vol"] = np.sqrt(d[model_col])

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=d["date"], y=d["realized_vol"],
        mode="lines", name="Realized Vol"
    ))

    fig.add_trace(go.Scatter(
        x=d["date"], y=d["forecast_vol"],
        mode="lines", name="Forecast Vol"
    ))

    fig.update_layout(
        title=f"{ticker}: Forecast vs Realized Volatility",
        xaxis_title="Date",
        yaxis_title="Volatility"
    )

    fig.show()

In [15]:
plot_vol_forecast(df_f, "SPY", "ewma1_var", "rv1_var")

In [16]:
plot_vol_forecast(df_f, "SPY", "hv1_var", "rv1_var")

In [17]:
''' Zooming-in to the crisis'''

def plot_zoom(df_f, ticker, model_col, target_col, start, end):
    d = df_f[df_f["ticker"] == ticker].copy()
    d = d[(d["date"] >= start) & (d["date"] <= end)].sort_values("date")

    d["realized_vol"] = np.sqrt(d[target_col])
    d["forecast_vol"] = np.sqrt(d[model_col])

    fig = go.Figure()

    fig.add_trace(go.Scatter(x=d["date"], y=d["realized_vol"], name="Realized"))
    fig.add_trace(go.Scatter(x=d["date"], y=d["forecast_vol"], name="Forecast"))

    fig.update_layout(
        title=f"{ticker} Crisis Zoom",
        xaxis_title="Date",
        yaxis_title="Volatility"
    )

    fig.show()

In [18]:
plot_zoom(df_f, "SPY", "ewma1_var", "rv1_var",
          "2008-01-01", "2009-01-01")

In [19]:
''' Plotting forecast errors to clearly see systematic underprediction during crisis onset and overshoot after shock subsides'''

def plot_forecast_error_vol(df_f, ticker, horizon=1, start=None, end=None, crisis_windows=None):
    """
    Plot forecast error in volatility space:
        error_vol = sqrt(forecast_var) - sqrt(realized_var)

   """
    d = df_f[df_f["ticker"] == ticker].sort_values("date").copy()

    # Optionally zoom
    if start is not None:
        d = d[d["date"] >= pd.to_datetime(start)]
    if end is not None:
        d = d[d["date"] <= pd.to_datetime(end)]

    target = f"rv{horizon}_var"
    hv_col = f"hv{horizon}_var"
    ewma_col = f"ewma{horizon}_var"

    # Convert variance -> volatility for interpretability
    d["realized_vol"] = np.sqrt(d[target])
    d["hv_vol"] = np.sqrt(d[hv_col])
    d["ewma_vol"] = np.sqrt(d[ewma_col])

    d["hv_err_vol"] = d["hv_vol"] - d["realized_vol"]
    d["ewma_err_vol"] = d["ewma_vol"] - d["realized_vol"]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=d["date"], y=d["hv_err_vol"], mode="lines", name=f"HV{horizon} error (vol)"))
    fig.add_trace(go.Scatter(x=d["date"], y=d["ewma_err_vol"], mode="lines", name=f"EWMA{horizon} error (vol)"))

    # Zero line
    fig.add_hline(y=0, line_width=1)

    fig.update_layout(
        title=f"{ticker}: Forecast Error in Vol Space (h={horizon})",
        xaxis_title="Date",
        yaxis_title="Forecast Vol − Realized Vol"
    )

    # Optional crisis shading
    if crisis_windows is not None:
        for name, (cs, ce) in crisis_windows.items():
            fig.add_vrect(
                x0=pd.to_datetime(cs), x1=pd.to_datetime(ce),
                fillcolor="gray", opacity=0.3, line_width=0,
                annotation_text=name, annotation_position="top left"
            )

    fig.show()

In [20]:
plot_forecast_error_vol(df_f, "SPY", horizon=1, crisis_windows=CRISIS_WINDOWS)
plot_forecast_error_vol(df_f, "JPM", horizon=1, crisis_windows=CRISIS_WINDOWS)


In [21]:
# Crisis zoom:
plot_forecast_error_vol(df_f, "SPY", horizon=1, start="2007-07-01", end="2009-06-30", crisis_windows={"GFC_2007_2009": ("2007-07-01", "2009-06-30") }) 
plot_forecast_error_vol(df_f, "SPY", horizon=1, start="2020-02-15", end="2020-05-31", crisis_windows={"COVID_2020": ("2020-02-15", "2020-05-31")})

As we can see in the plots above, the forecasted volatility spikes are smaller than the realized ones for both HV and EWMA models.

* Rolling HV : Uses past window.
  If today volatility explodes, it only incorporates yesterday's info and takes several days to fully reflect the new regime.

* EWMA: Uses recursive smoothing.
  Even if returns explode, lambda = 0.94 means 94% weight is on previous variance so it cannot instantly jump to crisis levels.

Therefore, during sudden regime shifts, forecasted variance lags behind realized variance.


This is the core limitation of linear volatility models. They assume
* gradual volatility evolution
* smooth clustering
* no structural breaks

But crises are structural breaks. That is why spikes look shorted in forecasts.

QLIKE results still look reasonable even though spikes are underestimated because:
* crisis periods are relatively short compared to calm periods
* QLIKE averages across entire sample
* mild underestimation doesn't dominate the metric.